# 🛡️ NetworkSearchAI — детектирование сетевых атак на лог-файлах

**Цель проекта:** обучить и сравнить несколько ML-моделей (MLP / RandomForest / XGBoost / LightGBM),
которые по записи о сетевом соединении в формате **Zeek `conn.log`** определяют, относится ли она к
одной из 6 категорий: `Normal`, `Spoofing`, `Suspicious`, `Flooding`, `Brute-Force`, `DDoS`.

**Что нового по сравнению с базовой версией:**
1. Воспроизводимость — фиксированные seed-ы для `random`, `numpy`, `tensorflow`.
2. Корректная разметка — явная иерархия меток (от слабой к сильной).
3. Расширенный feature engineering — агрегаты по `src_ip`, временные признаки, флаги.
4. Нет утечки данных — `StandardScaler` обучается только на train.
5. Полноценный MLP — `Dropout`, `BatchNormalization`, L2, `ReduceLROnPlateau`, `ModelCheckpoint`.
6. Сравнение 4 моделей с единой схемой оценки.
7. Расширенная оценка — ROC-AUC, PR-AUC, permutation importance, анализ ошибок.
8. Inference-функция: принимает новую запись → возвращает класс и вероятности.
9. Все артефакты сохраняются (`model.keras`, `scaler.pkl`, `label_encoder.pkl`, `feature_names.json`).
10. Запускается **без оригинального `conn.log`** — встроенный генератор синтетических данных.

---

> **Среда выполнения.** Можно запускать локально (`pip install -r requirements.txt`)
> или в Google Colab. На полных данных (десятки миллионов строк) рекомендуется среда
> с большим RAM или TPU/GPU. Синтетический режим работает на обычном CPU за минуты.


## 1. Установка и импорт библиотек

In [ ]:
# Раскомментируйте в Colab. Локально лучше использовать requirements.txt
# !pip install numpy pandas scikit-learn tensorflow matplotlib seaborn xgboost lightgbm joblib


In [ ]:
import os
import json
import random
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, f1_score, accuracy_score,
)
from sklearn.inspection import permutation_importance

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks

# XGBoost / LightGBM — необязательно, оборачиваем в try/except
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("⚠️  xgboost не установлен — соответствующая модель будет пропущена")

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("⚠️  lightgbm не установлен — соответствующая модель будет пропущена")

import joblib

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
print(f"TF version: {tf.__version__}")


### Конфигурация и воспроизводимость

In [ ]:
# ---- CONFIG ----
CFG = {
    # Пути
    "log_path":          "conn.log",       # путь к Zeek conn.log
    "artifacts_dir":     "artifacts",      # куда сохраняем модели / scaler / encoder
    "use_synthetic":     "auto",           # "auto" | True | False
    "synthetic_size":    25_000,           # сколько Normal-строк сгенерировать

    # Разметка
    "rules": {
        "ddos_per_day":       1000,
        "brute_force_sess":   50,
        "brute_force_ports":  (5, 30),     # focused, не сканер
        "suspicious_bytes":   1e9,
        "flooding_ports":     500,
    },

    # Обучение
    "test_size":         0.2,
    "val_size":          0.2,              # из train выделяется на валидацию
    "random_state":      42,
    "epochs":            40,
    "batch_size":        256,
    "patience":          7,
}

SEED = CFG["random_state"]
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

Path(CFG["artifacts_dir"]).mkdir(exist_ok=True, parents=True)


## 2. Загрузка данных

Если файл `conn.log` доступен — он будет прочитан. Если нет — генерируется
**синтетический набор** в том же 20-колонковом формате, с реалистичными паттернами
для каждого типа атаки. Это позволяет запустить весь пайплайн без оригинальных данных.


In [ ]:
COLUMNS_FULL = [
    "timestamp", "session_id", "src_ip", "src_port", "dst_ip", "dst_port",
    "protocol", "app_protocol", "duration", "bytes_sent", "bytes_received",
    "packets_sent", "packets_received", "flags", "unknown1", "unknown2",
    "unknown3", "unknown4", "unknown5", "additional_info",
]
COLUMNS_KEEP = COLUMNS_FULL[:13]   # первые 13 информативных


def load_real_log(path: str) -> pd.DataFrame:
    df = pd.read_csv(
        path, sep="\t", header=None, names=COLUMNS_FULL,
        na_values=["(empty)", "-"], on_bad_lines="skip",
        low_memory=False, comment="#",
    )
    return df[COLUMNS_KEEP]


def generate_synthetic(n_normal: int = 25_000, base_ts: int = 1_700_000_000) -> pd.DataFrame:
    # Generate a synthetic Zeek-style conn.log with embedded attack patterns.
    rng = random.Random(SEED)

    def rip(prefix=None):
        if prefix:
            fixed = prefix.split(".")
            rest = [str(rng.randint(1, 254)) for _ in range(4 - len(fixed))]
            return ".".join(fixed + rest)
        return ".".join(str(rng.randint(1, 254)) for _ in range(4))

    rows = []
    protocols = ["tcp", "udp", "icmp"]
    apps = ["http", "https", "dns", "ssh", "ftp", "smtp", "-"]

    # Normal
    for _ in range(n_normal):
        rows.append([
            base_ts + rng.randint(0, 86400 * 7),
            f"C{rng.randint(1, 99999999):08d}",
            rip(), rng.randint(1024, 65535), rip(),
            rng.choice([80, 443, 22, 53, 25, 8080]),
            rng.choice(protocols), rng.choice(apps),
            round(rng.uniform(0.01, 30.0), 3),
            rng.randint(50, 50_000), rng.randint(50, 50_000),
            rng.randint(1, 50), rng.randint(1, 50),
        ])

    # Brute-Force
    for _ in range(3):
        bf_ip = rip("10.0.5")
        for _ in range(200):
            rows.append([
                base_ts + rng.randint(0, 86400 * 7),
                f"C{rng.randint(1, 99999999):08d}",
                bf_ip, rng.randint(1024, 65535), rip(),
                rng.choice([22, 23, 3389, 21, 445, 5900, 1433, 3306]),
                "tcp", rng.choice(["ssh", "-"]),
                round(rng.uniform(0.01, 2.0), 3),
                rng.randint(80, 500), rng.randint(0, 200),
                rng.randint(1, 5), rng.randint(0, 5),
            ])

    # DDoS
    for _ in range(2):
        ddos_ip = rip("172.16.4")
        day_start = base_ts + 86400 * rng.randint(0, 6)
        for _ in range(1300):
            rows.append([
                day_start + rng.randint(0, 86400),
                f"C{rng.randint(1, 99999999):08d}",
                ddos_ip, rng.randint(1024, 65535), rip(), 80,
                "tcp", "http",
                round(rng.uniform(0.001, 0.1), 3),
                rng.randint(20, 200), rng.randint(0, 100),
                rng.randint(1, 3), rng.randint(0, 3),
            ])

    # Suspicious (huge byte transfers)
    for _ in range(4):
        susp_ip = rip("203.0.113")
        for _ in range(50):
            rows.append([
                base_ts + rng.randint(0, 86400 * 7),
                f"C{rng.randint(1, 99999999):08d}",
                susp_ip, rng.randint(1024, 65535), rip(),
                rng.choice([443, 8443, 9000, 50000]),
                "tcp", "https",
                round(rng.uniform(10, 600), 3),
                rng.randint(50_000_000, 200_000_000),
                rng.randint(50_000_000, 200_000_000),
                rng.randint(1000, 5000), rng.randint(1000, 5000),
            ])

    # Flooding (port scan)
    for _ in range(2):
        fl_ip = rip("198.51.100")
        for port in rng.sample(range(1, 65535), 650):
            rows.append([
                base_ts + rng.randint(0, 86400 * 7),
                f"C{rng.randint(1, 99999999):08d}",
                fl_ip, rng.randint(1024, 65535), rip(), port,
                "tcp", "-",
                round(rng.uniform(0.001, 0.5), 3),
                rng.randint(40, 200), rng.randint(0, 100),
                rng.randint(1, 3), rng.randint(0, 3),
            ])

    # Spoofing (internal-IP sources)
    for _ in range(800):
        rows.append([
            base_ts + rng.randint(0, 86400 * 7),
            f"C{rng.randint(1, 99999999):08d}",
            rip("192.168.1"), rng.randint(1024, 65535), rip(),
            rng.choice([80, 443, 53]),
            rng.choice(protocols), rng.choice(apps),
            round(rng.uniform(0.01, 10.0), 3),
            rng.randint(50, 20_000), rng.randint(50, 20_000),
            rng.randint(1, 20), rng.randint(1, 20),
        ])

    df = pd.DataFrame(rows, columns=COLUMNS_KEEP)
    return df.sample(frac=1, random_state=SEED).reset_index(drop=True)


In [ ]:
# ---- Решаем, что грузить ----
use_synth = CFG["use_synthetic"]
if use_synth == "auto":
    use_synth = not Path(CFG["log_path"]).exists()

if use_synth:
    print("📦 Используется синтетический набор (conn.log не найден или режим явно включён)")
    df = generate_synthetic(n_normal=CFG["synthetic_size"])
else:
    print(f"📂 Чтение реального лога: {CFG['log_path']}")
    df = load_real_log(CFG["log_path"])

print(f"Размер DataFrame: {df.shape}")
df.head()


## 3. Очистка и подготовка

In [ ]:
# timestamp → datetime; числовые колонки → numeric
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", errors="coerce")

NUMERIC_COLS = ["duration", "bytes_sent", "bytes_received",
                "packets_sent", "packets_received",
                "src_port", "dst_port"]
for c in NUMERIC_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Категориальные — оставляем как строки, но приводим к нижнему регистру и заполняем пропуски
for c in ["protocol", "app_protocol"]:
    df[c] = df[c].astype(str).str.lower().replace({"nan": "unknown", "none": "unknown"}).fillna("unknown")

print("Пропуски до заполнения:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Числовые пропуски → 0 (после to_numeric)
df[NUMERIC_COLS] = df[NUMERIC_COLS].fillna(0)

# Строки без timestamp удаляем — они бесполезны для временных признаков
before = len(df)
df = df.dropna(subset=["timestamp"]).reset_index(drop=True)
print(f"\nУдалено строк без timestamp: {before - len(df)}")
print(f"Итог: {df.shape}")


## 4. Правило-ориентированная разметка

Применяем правила в порядке **от слабого к сильному** — таким образом более серьёзный тип атаки
переписывает более лёгкий. Иерархия:

`Normal` → `Spoofing` → `Suspicious` → `Flooding` → `Brute-Force` → `DDoS`


In [ ]:
def label_data(df: pd.DataFrame, rules: dict) -> pd.DataFrame:
    df = df.copy()
    df["label"] = "Normal"

    # --- Spoofing: внутренние IP (RFC1918 + link-local IPv6) ---
    internal_re = r"^(10\.|192\.168\.|172\.(1[6-9]|2\d|3[01])\.|fe80::)"
    df.loc[df["src_ip"].astype(str).str.match(internal_re), "label"] = "Spoofing"

    # --- Suspicious: огромный суммарный трафик c одного IP ---
    totals = df.groupby("src_ip")[["bytes_sent", "bytes_received"]].sum().sum(axis=1)
    susp_ips = totals[totals > rules["suspicious_bytes"]].index
    df.loc[df["src_ip"].isin(susp_ips), "label"] = "Suspicious"

    # --- Flooding: >N уникальных dst_port с одного IP (port scan) ---
    n_ports_per_ip = df.groupby("src_ip")["dst_port"].nunique()
    flood_ips = n_ports_per_ip[n_ports_per_ip > rules["flooding_ports"]].index
    df.loc[df["src_ip"].isin(flood_ips), "label"] = "Flooding"

    # --- Brute-Force: много сессий, но узкий диапазон портов ---
    bf_grp = df.groupby("src_ip").agg(
        n_sess=("session_id", "nunique"),
        n_ports=("dst_port", "nunique"),
    )
    lo, hi = rules["brute_force_ports"]
    bf_ips = bf_grp[(bf_grp["n_sess"] > rules["brute_force_sess"])
                    & (bf_grp["n_ports"] > lo)
                    & (bf_grp["n_ports"] < hi)].index
    df.loc[df["src_ip"].isin(bf_ips), "label"] = "Brute-Force"

    # --- DDoS: >N подключений за сутки с одного IP ---
    df["_date"] = df["timestamp"].dt.date
    daily = df.groupby(["src_ip", "_date"]).size().reset_index(name="cnt")
    ddos_ips = daily.loc[daily["cnt"] > rules["ddos_per_day"], "src_ip"].unique()
    df.loc[df["src_ip"].isin(ddos_ips), "label"] = "DDoS"
    df = df.drop(columns="_date")

    return df


df = label_data(df, CFG["rules"])
print("Распределение классов:")
print(df["label"].value_counts())


## 5. Разведочный анализ (EDA)

In [ ]:
# Распределение классов
plt.figure(figsize=(10, 5))
order = df["label"].value_counts().index.tolist()
ax = sns.countplot(data=df, x="label", order=order, hue="label",
                   palette="viridis", legend=False)
plt.title("Распределение классов")
plt.xlabel("Класс"); plt.ylabel("Количество")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout(); plt.show()


In [ ]:
# Top-10 IP по количеству подключений
top_ips = df["src_ip"].value_counts().head(10)
plt.figure(figsize=(12, 6))
sns.barplot(x=top_ips.values, y=top_ips.index, hue=top_ips.index,
            palette="viridis", legend=False)
plt.title("Top-10 IP по количеству подключений")
plt.xlabel("Подключения"); plt.ylabel("IP")
for i, v in enumerate(top_ips.values):
    plt.text(v, i, f"  {v}", va="center")
plt.tight_layout(); plt.show()


In [ ]:
# Активность по часам
df["hour"] = df["timestamp"].dt.hour
hourly = df.groupby("hour").size()
plt.figure(figsize=(13, 5))
sns.barplot(x=hourly.index, y=hourly.values, hue=hourly.index,
            palette="Blues_d", legend=False)
plt.title("Активность по часам суток")
plt.xlabel("Час"); plt.ylabel("Подключения")
plt.tight_layout(); plt.show()


In [ ]:
# Top-15 IP по объёму трафика
df["total_bytes"] = df["bytes_sent"] + df["bytes_received"]
top_bytes = (df.groupby("src_ip")["total_bytes"].sum()
               .sort_values(ascending=False).head(15))
plt.figure(figsize=(13, 7))
sns.barplot(x=top_bytes.values, y=top_bytes.index, hue=top_bytes.index,
            palette="coolwarm", legend=False)
plt.title("Top-15 IP по объёму трафика")
plt.xlabel("Байты"); plt.ylabel("IP")
plt.tight_layout(); plt.show()


In [ ]:
# Тепловая карта корреляций числовых признаков
plt.figure(figsize=(9, 7))
corr = df[NUMERIC_COLS + ["total_bytes"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Корреляции числовых признаков")
plt.tight_layout(); plt.show()


## 6. Feature Engineering

Расширяем признаковое пространство:

| Группа | Признаки |
|---|---|
| Объёмные | `total_bytes`, `bytes_ratio`, `avg_pkt_size_sent`, `avg_pkt_size_recv` |
| Бинарные | `is_internal`, `is_well_known_port`, `is_ephemeral` |
| Временные | `hour`, `dayofweek`, `is_weekend` |
| Агрегаты по `src_ip` | `src_n_sess`, `src_n_dst_ports`, `src_n_dst_ips`, `src_total_bytes`, `src_mean_dur`, `src_port_entropy` |


In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # --- ratios / volumes ---
    df["total_bytes"] = df["bytes_sent"] + df["bytes_received"]
    df["bytes_ratio"] = df["bytes_sent"] / (df["bytes_received"] + 1.0)
    df["avg_pkt_size_sent"] = df["bytes_sent"] / (df["packets_sent"] + 1.0)
    df["avg_pkt_size_recv"] = df["bytes_received"] / (df["packets_received"] + 1.0)

    # --- binary flags ---
    internal_re = r"^(10\.|192\.168\.|172\.(1[6-9]|2\d|3[01])\.)"
    df["is_internal"] = df["src_ip"].astype(str).str.match(internal_re).astype(int)
    df["is_well_known_port"] = (df["dst_port"] < 1024).astype(int)
    df["is_ephemeral"] = (df["src_port"] >= 49152).astype(int)

    # --- time ---
    df["hour"] = df["timestamp"].dt.hour
    df["dayofweek"] = df["timestamp"].dt.dayofweek
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

    # --- per-src_ip aggregates ---
    def port_entropy(s):
        p = s.value_counts(normalize=True).values
        return -np.sum(p * np.log2(p + 1e-12))

    agg = df.groupby("src_ip").agg(
        src_n_sess=("session_id", "nunique"),
        src_n_dst_ports=("dst_port", "nunique"),
        src_n_dst_ips=("dst_ip", "nunique"),
        src_total_bytes=("total_bytes", "sum"),
        src_mean_dur=("duration", "mean"),
        src_port_entropy=("dst_port", port_entropy),
    ).reset_index()
    df = df.merge(agg, on="src_ip", how="left")
    return df


df = add_features(df)
print("Колонки после feature engineering:")
print(df.columns.tolist())


## 7. Балансировка классов

Класс `Normal` обычно доминирует. Чтобы не «съедал» обучение и метрики, делаем undersampling:
число `Normal` урезаем до медианного размера атак, а малые классы оставляем как есть.


In [ ]:
def balance_classes(df: pd.DataFrame, max_per_class: int = None,
                    random_state: int = SEED) -> pd.DataFrame:
    counts = df["label"].value_counts()
    attack_counts = counts.drop("Normal", errors="ignore")
    median_attack = int(attack_counts.median()) if len(attack_counts) else len(df)

    if max_per_class is None:
        max_per_class = max(median_attack * 2, 1000)

    parts = []
    for lbl, sub in df.groupby("label"):
        n = min(len(sub), max_per_class)
        parts.append(sub.sample(n=n, random_state=random_state))
    out = (pd.concat(parts)
             .sample(frac=1, random_state=random_state)
             .reset_index(drop=True))
    return out


balanced_df = balance_classes(df)
print("После балансировки:")
print(balanced_df["label"].value_counts())


## 8. Подготовка признаков для моделей

In [ ]:
# Признаки и целевая переменная
FEATURES_NUM = [
    "src_port", "dst_port", "duration",
    "bytes_sent", "bytes_received", "packets_sent", "packets_received",
    "total_bytes", "bytes_ratio", "avg_pkt_size_sent", "avg_pkt_size_recv",
    "is_internal", "is_well_known_port", "is_ephemeral",
    "hour", "dayofweek", "is_weekend",
    "src_n_sess", "src_n_dst_ports", "src_n_dst_ips",
    "src_total_bytes", "src_mean_dur", "src_port_entropy",
]
FEATURES_CAT = ["protocol", "app_protocol"]

X = balanced_df[FEATURES_NUM + FEATURES_CAT].copy()
X = pd.get_dummies(X, columns=FEATURES_CAT, drop_first=True)

# Заполнение возможных NaN после фичей (например, log/деление на 0)
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

y_raw = balanced_df["label"].values

le = LabelEncoder()
y = le.fit_transform(y_raw)
N_CLASSES = len(le.classes_)
print("Классы:", list(le.classes_))
print("Размер X:", X.shape)


In [ ]:
# --- Train / Val / Test split ---
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X.values, y, test_size=CFG["test_size"], random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=CFG["val_size"], random_state=SEED, stratify=y_tmp
)
print(f"train={X_train.shape}  val={X_val.shape}  test={X_test.shape}")

# --- Scaling: fit ТОЛЬКО на train! ---
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)


## 9. Модель MLP (TensorFlow / Keras)

Архитектура: 4 полносвязных слоя с `BatchNormalization`, `Dropout`, L2-регуляризацией.
Колбэки: `EarlyStopping`, `ReduceLROnPlateau`, `ModelCheckpoint`.


In [ ]:
def build_mlp(input_dim: int, n_classes: int, l2: float = 1e-4) -> tf.keras.Model:
    m = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(l2)),
        layers.BatchNormalization(),
        layers.Dropout(0.4),

        layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(l2)),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(l2)),
        layers.BatchNormalization(),
        layers.Dropout(0.2),

        layers.Dense(32, activation="relu"),
        layers.Dense(n_classes, activation="softmax"),
    ])
    m.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return m


mlp = build_mlp(X_train_s.shape[1], N_CLASSES)
mlp.summary()


In [ ]:
# Веса классов (на train)
cls_w = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights = dict(zip(np.unique(y_train), cls_w))
print("Веса классов:", class_weights)

ckpt_path = Path(CFG["artifacts_dir"]) / "mlp_best.keras"

cb = [
    callbacks.EarlyStopping(monitor="val_loss", patience=CFG["patience"],
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                patience=3, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(filepath=str(ckpt_path), monitor="val_loss",
                              save_best_only=True, verbose=0),
]

history = mlp.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=CFG["epochs"],
    batch_size=CFG["batch_size"],
    class_weight=class_weights,
    callbacks=cb,
    verbose=2,
)


In [ ]:
# Кривые обучения
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].set_title("Accuracy"); axes[0].set_xlabel("epoch"); axes[0].legend()

axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("Loss"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()


## 10. Baseline-модели: RandomForest, XGBoost, LightGBM

На табличных данных градиентный бустинг и случайный лес часто конкурентоспособны с MLP.
Сравниваем их в одинаковых условиях.


In [ ]:
trained = {}

# --- RandomForest ---
rf = RandomForestClassifier(
    n_estimators=300, max_depth=None, n_jobs=-1,
    class_weight="balanced", random_state=SEED,
)
rf.fit(X_train_s, y_train)
trained["RandomForest"] = rf

# --- XGBoost ---
if HAS_XGB:
    xgb_clf = xgb.XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.1,
        subsample=0.9, colsample_bytree=0.9,
        objective="multi:softprob", num_class=N_CLASSES,
        eval_metric="mlogloss", random_state=SEED,
        tree_method="hist", n_jobs=-1,
    )
    xgb_clf.fit(X_train_s, y_train,
                eval_set=[(X_val_s, y_val)], verbose=False)
    trained["XGBoost"] = xgb_clf

# --- LightGBM ---
if HAS_LGB:
    lgb_clf = lgb.LGBMClassifier(
        n_estimators=400, num_leaves=63, learning_rate=0.05,
        class_weight="balanced", objective="multiclass",
        num_class=N_CLASSES, random_state=SEED, n_jobs=-1,
        verbose=-1,
    )
    lgb_clf.fit(X_train_s, y_train,
                eval_set=[(X_val_s, y_val)])
    trained["LightGBM"] = lgb_clf

print("Обучены:", list(trained.keys()))


## 11. Единая система оценки

In [ ]:
def predict_proba(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)
    return model.predict(X, verbose=0)  # Keras


def evaluate(model, X_test, y_test, name: str):
    proba = predict_proba(model, X_test)
    y_pred = np.argmax(proba, axis=1)

    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    try:
        roc = roc_auc_score(y_test, proba, multi_class="ovr", average="macro")
    except Exception:
        roc = float("nan")
    try:
        pr = average_precision_score(
            np.eye(N_CLASSES)[y_test], proba, average="macro"
        )
    except Exception:
        pr = float("nan")

    return {
        "model": name,
        "accuracy": acc,
        "macro_f1": f1_macro,
        "weighted_f1": f1_weighted,
        "macro_roc_auc": roc,
        "macro_pr_auc": pr,
        "y_pred": y_pred,
        "proba": proba,
    }


# MLP в общий пул
trained_all = {"MLP": mlp, **trained}

results = []
for name, model in trained_all.items():
    r = evaluate(model, X_test_s, y_test, name)
    results.append(r)
    print(f"\n=== {name} ===")
    print(f"accuracy={r['accuracy']:.4f}  macro_f1={r['macro_f1']:.4f}  "
          f"roc_auc={r['macro_roc_auc']:.4f}  pr_auc={r['macro_pr_auc']:.4f}")


In [ ]:
# Сравнение моделей в виде таблицы
cmp_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ("y_pred", "proba")}
                       for r in results])
cmp_df = cmp_df.set_index("model").round(4)
display(cmp_df)


In [ ]:
# Сравнение моделей: bar-chart
melted = cmp_df.reset_index().melt(id_vars="model", var_name="metric", value_name="score")
plt.figure(figsize=(11, 6))
sns.barplot(data=melted, x="model", y="score", hue="metric", palette="viridis")
plt.title("Сравнение моделей по ключевым метрикам")
plt.ylim(max(0, melted["score"].min() - 0.05), 1.0)
plt.xticks(rotation=0); plt.legend(loc="lower right")
plt.tight_layout(); plt.show()


## 12. Детальный разбор лучшей модели

Берём модель с наивысшим `macro_f1` и смотрим на неё подробно.


In [ ]:
best_idx = int(np.argmax([r["macro_f1"] for r in results]))
best_name = results[best_idx]["model"]
best_pred = results[best_idx]["y_pred"]
best_proba = results[best_idx]["proba"]
best_model = trained_all[best_name]

print(f"🏆 Лучшая модель: {best_name}")
print(classification_report(y_test, best_pred, target_names=le.classes_, zero_division=0))


In [ ]:
# Матрица путаницы — абсолютная и нормализованная
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cm = confusion_matrix(y_test, best_pred)
cm_n = confusion_matrix(y_test, best_pred, normalize="true")

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[0])
axes[0].set_title(f"Confusion matrix — {best_name}")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

sns.heatmap(cm_n, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[1])
axes[1].set_title("Confusion matrix (normalized)")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
plt.tight_layout(); plt.show()


In [ ]:
# ROC-кривые для каждого класса (One-vs-Rest)
plt.figure(figsize=(9, 7))
y_test_oh = np.eye(N_CLASSES)[y_test]
for i, cls_name in enumerate(le.classes_):
    fpr, tpr, _ = roc_curve(y_test_oh[:, i], best_proba[:, i])
    auc_i = roc_auc_score(y_test_oh[:, i], best_proba[:, i])
    plt.plot(fpr, tpr, label=f"{cls_name}  AUC={auc_i:.3f}")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.title(f"ROC curves (One-vs-Rest) — {best_name}")
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.legend(loc="lower right")
plt.tight_layout(); plt.show()


In [ ]:
# Precision-Recall кривые
plt.figure(figsize=(9, 7))
for i, cls_name in enumerate(le.classes_):
    prec, rec, _ = precision_recall_curve(y_test_oh[:, i], best_proba[:, i])
    ap_i = average_precision_score(y_test_oh[:, i], best_proba[:, i])
    plt.plot(rec, prec, label=f"{cls_name}  AP={ap_i:.3f}")
plt.title(f"Precision-Recall curves — {best_name}")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.legend(loc="lower left")
plt.tight_layout(); plt.show()


In [ ]:
# Permutation importance (на подвыборке для скорости)
sample_n = min(2000, X_test_s.shape[0])
idx = np.random.RandomState(SEED).choice(X_test_s.shape[0], sample_n, replace=False)

# Permutation importance работает с любым sklearn-совместимым API,
# поэтому для MLP оборачиваем в простой адаптер
class KerasAdapter:
    def __init__(self, model): self.model = model
    def fit(self, X, y): return self
    def predict(self, X): return np.argmax(self.model.predict(X, verbose=0), axis=1)
    def score(self, X, y): return accuracy_score(y, self.predict(X))

est = KerasAdapter(best_model) if best_name == "MLP" else best_model

pi = permutation_importance(est, X_test_s[idx], y_test[idx],
                            n_repeats=5, random_state=SEED, n_jobs=-1)

imp_df = (pd.DataFrame({"feature": X.columns, "importance": pi.importances_mean})
            .sort_values("importance", ascending=True))

plt.figure(figsize=(10, max(6, 0.32 * len(imp_df))))
sns.barplot(data=imp_df, x="importance", y="feature", hue="feature",
            palette="viridis", legend=False)
plt.title(f"Permutation importance — {best_name}")
plt.tight_layout(); plt.show()


### Анализ ошибок

In [ ]:
# Какие пары классов путаются сильнее всего?
errors = (pd.DataFrame({
    "true": le.inverse_transform(y_test),
    "pred": le.inverse_transform(best_pred),
})
.query("true != pred")
.groupby(["true", "pred"]).size()
.reset_index(name="count")
.sort_values("count", ascending=False))

print("Топ-10 типов ошибок:")
print(errors.head(10).to_string(index=False))


## 13. Сохранение артефактов

Сохраняем всё, что нужно для inference в проде: модель, scaler, encoder и список фичей.


In [ ]:
art = Path(CFG["artifacts_dir"])

# 1) Лучшая модель
if best_name == "MLP":
    best_model.save(art / "best_mlp.keras")
else:
    joblib.dump(best_model, art / f"best_{best_name.lower()}.pkl")

# 2) Все обученные модели — на случай переанализа
for nm, m in trained.items():
    joblib.dump(m, art / f"model_{nm.lower()}.pkl")
mlp.save(art / "model_mlp.keras")

# 3) Препроцессоры
joblib.dump(scaler, art / "scaler.pkl")
joblib.dump(le, art / "label_encoder.pkl")

# 4) Метаданные
meta = {
    "saved_at": datetime.now().isoformat(timespec="seconds"),
    "best_model": best_name,
    "classes": list(le.classes_),
    "features": list(X.columns),
    "metrics": cmp_df.reset_index().to_dict(orient="records"),
    "config": {k: v for k, v in CFG.items() if k != "rules"} | {"rules": CFG["rules"]},
}
with open(art / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2, default=str)

print(f"✅ Артефакты сохранены в: {art.resolve()}")
for p in sorted(art.iterdir()):
    print(f"   {p.name}")


## 14. Inference: предсказание для новой записи

Функция-помощник: принимает обычный dict (как одна строка лога) и возвращает
название класса + вероятности.


In [ ]:
def predict_one(record: dict, model=None, scaler=scaler, le=le,
                feature_columns=list(X.columns)):
    # record — словарь с полями строки conn.log.
    # Возвращает (class_name, probabilities_dict).
    model = model or best_model
    row = pd.DataFrame([record])

    # Минимальная очистка
    row["timestamp"] = pd.to_datetime(row["timestamp"], unit="s", errors="coerce")
    for c in NUMERIC_COLS:
        row[c] = pd.to_numeric(row[c], errors="coerce")
    row = row.fillna(0)
    for c in ["protocol", "app_protocol"]:
        row[c] = row[c].astype(str).str.lower()

    # Те же фичи, что при обучении. Для агрегатов одной строки используем нули —
    # в проде их надо подмешивать из бэкграунд-стейта по src_ip.
    row = add_features(row)
    for c in ["src_n_sess", "src_n_dst_ports", "src_n_dst_ips",
              "src_total_bytes", "src_mean_dur", "src_port_entropy"]:
        if row[c].isna().any():
            row[c] = 0

    X_one = pd.get_dummies(row[FEATURES_NUM + FEATURES_CAT],
                           columns=FEATURES_CAT, drop_first=True)
    X_one = X_one.reindex(columns=feature_columns, fill_value=0)
    X_one = X_one.replace([np.inf, -np.inf], np.nan).fillna(0).values
    X_one = scaler.transform(X_one)

    proba = predict_proba(model, X_one)[0]
    cls = le.inverse_transform([int(np.argmax(proba))])[0]
    return cls, dict(zip(le.classes_, proba.round(4)))


# Демонстрация: возьмём первую строку из тестового набора и сравним
demo_record = {
    "timestamp":        1_700_001_234,
    "session_id":       "Cabc12345",
    "src_ip":           "10.0.5.13",
    "src_port":         54321,
    "dst_ip":           "8.8.8.8",
    "dst_port":         22,
    "protocol":         "tcp",
    "app_protocol":     "ssh",
    "duration":         0.4,
    "bytes_sent":       150,
    "bytes_received":   30,
    "packets_sent":     3,
    "packets_received": 1,
}
cls, probs = predict_one(demo_record)
print(f"Предсказанный класс: {cls}")
print("Вероятности:")
for k, v in sorted(probs.items(), key=lambda x: -x[1]):
    print(f"  {k:12s} {v:.4f}")


## 15. Итоги

* Пайплайн end-to-end: загрузка → правило-ориентированная разметка → feature engineering →
  балансировка → обучение 4 моделей → детальная оценка → сохранение артефактов → inference.
* Воспроизводимость: фиксированные сиды, train/val/test split без утечки, scaler на train.
* Сравнили MLP с RandomForest / XGBoost / LightGBM по согласованным метрикам
  (accuracy, macro-F1, ROC-AUC, PR-AUC).
* Артефакты для прода: `best_*.keras`/`.pkl`, `scaler.pkl`, `label_encoder.pkl`, `metadata.json`.

**Что можно улучшить дальше:**
* Заменить правило-ориентированную разметку на размеченный датасет (CICIDS, UNSW-NB15).
* Скользящие агрегаты по `src_ip` за последние N минут вместо глобальных.
* Sequence-модель (1D CNN / Transformer) на последовательностях соединений одного IP.
* Stratified k-fold вместо одного split-а для устойчивых оценок.
* Калибровка вероятностей (`CalibratedClassifierCV`) для интерпретируемого порога.
* Деплой в виде сервиса (FastAPI + Docker).
